In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
from autogluon.tabular import TabularPredictor
from sklearn.metrics import average_precision_score

# ============================================================
# 1) 데이터 로드
# ------------------------------------------------------------
# - PANEL_PATH에 저장된 최종 패널 데이터를 불러온다.
# - 이 코드는 snapshot 단위로 생성된 패널 데이터를 입력으로 사용한다.
# - 프로젝트 기준 구조:
#   * snapshot cadence = 7일
#   * feature window   = 과거 28일
#   * label window     = 미래 14일
# - 즉 각 행은 특정 user_id의 특정 snapshot_date 시점 관측값이며,
#   churn_label은 그 시점 이후 14일 내 churn 여부를 나타낸다.
# ============================================================
df = pd.read_csv(PANEL_PATH)

# snapshot_date를 문자열이 아닌 datetime 타입으로 변환한다.
# 이후 시계열 분할(train / valid / test) 시 날짜 정렬과 비교에 사용된다.
df["snapshot_date"] = pd.to_datetime(df["snapshot_date"])

# 모델 학습용 타깃 컬럼을 생성한다.
# churn_label을 정수형(0/1)으로 변환하여 target이라는 표준 학습용 컬럼명으로 맞춘다.
df["target"] = df["churn_label"].astype(int)

# ============================================================
# 2) 최종 변수 10개
# ------------------------------------------------------------
# - 현재 공식 모델에 사용하는 최종 feature 목록이다.
# - 기존 결과 유지가 원칙이므로, feature 구성은 임의 변경하지 않는다.
# - 아래 10개는 최종 선정된 핵심 입력 변수이며, 모델 학습 및 대시보드 저장에 함께 사용된다.
# ============================================================
final_features = [
    "had_core_watch_history_rebuilt",  # 핵심 콘텐츠 시청 이력 존재 여부(재구축 버전)
    "watch_hours",                     # 총 시청 시간
    "content_diversity_score",         # 시청 콘텐츠 다양성 점수
    "price_score",                     # 가격 민감도/가격 관련 성향 점수
    "had_watch_delta_rebuilt",         # 최근 시청 변화 이력 존재 여부(재구축 버전)
    "days_since_last_watch",           # 마지막 시청 후 경과 일수
    "freq_smartphone",                 # 스마트폰 시청 비중/빈도
    "freq_tv_set",                     # TV 시청 비중/빈도
    "completion_rate",                 # 콘텐츠 완주율
    "search_engagement"                # 검색 행동 관여도
]

# 실제 데이터프레임에 존재하는 컬럼만 남긴다.
# 이유:
# - 패널 버전 차이 또는 전처리 상태 차이로 일부 컬럼이 없을 수 있기 때문
# - 없는 컬럼 때문에 코드가 중단되지 않도록 방어적으로 처리
final_features = [col for col in final_features if col in df.columns]

# 대시보드 보조 정보로 유지할 컬럼들이다.
# 이 컬럼들은 모델 입력에서는 제외하지만,
# 후속 결과 저장(dashboard_final)에는 포함될 수 있다.
extra_keep_cols = [
    "segment_volume", "segment_explore", "segment_taste",
    "freq_tablet", "freq_pc"
]

# 마찬가지로 실제 존재하는 컬럼만 남긴다.
extra_keep_cols = [col for col in extra_keep_cols if col in df.columns]

# 모델링/평가/대시보드용 최소 데이터셋을 구성한다.
# 포함 항목:
# - 식별용: user_id, snapshot_date
# - 타깃: target
# - 모델 입력: final_features
# - 보조 보존 컬럼: extra_keep_cols
model_df = df[["user_id", "snapshot_date", "target"] + final_features + extra_keep_cols].copy()

# ============================================================
# 3) 시계열 분할
# ------------------------------------------------------------
# - snapshot_date 기준으로 시계열 분할을 수행한다.
# - 랜덤 분할이 아니라 시간 순서를 유지하는 방식이다.
# - 가장 최근 2개 snapshot 중:
#   * 마지막 이전 snapshot = validation
#   * 가장 마지막 snapshot = test
# - 그 이전 snapshot 전체 = train
# - 미래 정보 누수를 막기 위한 기본 구조다.
# ============================================================
snapshot_dates = sorted(model_df["snapshot_date"].dropna().unique())

# 마지막 2개 snapshot을 제외한 과거 구간을 학습용으로 사용
train_dates = snapshot_dates[:-2]

# 마지막에서 두 번째 snapshot을 검증용으로 사용
valid_date = snapshot_dates[-2]

# 가장 마지막 snapshot을 최종 테스트용으로 사용
test_date = snapshot_dates[-1]

# 각 날짜 구간별 데이터셋 분리
train_data = model_df[model_df["snapshot_date"].isin(train_dates)].copy()
valid_data = model_df[model_df["snapshot_date"] == valid_date].copy()
test_data  = model_df[model_df["snapshot_date"] == test_date].copy()

# 실제 모델 입력용 데이터셋 생성
# 제거 대상:
# - user_id: 식별자이며 모델 입력에 직접 사용하지 않음
# - snapshot_date: 시점 분할용 컬럼이며 모델 입력에서 제외
# - extra_keep_cols: 대시보드/후처리용 유지 컬럼이므로 학습 입력에서는 제거
# errors="ignore"를 사용하여 없는 컬럼이 있어도 에러 없이 통과하도록 설정
train_model = train_data.drop(columns=["user_id", "snapshot_date"] + extra_keep_cols, errors="ignore")
valid_model = valid_data.drop(columns=["user_id", "snapshot_date"] + extra_keep_cols, errors="ignore")
test_model  = test_data.drop(columns=["user_id", "snapshot_date"] + extra_keep_cols, errors="ignore")

# ============================================================
# 4) 평가 함수
# ------------------------------------------------------------
# - PR-AUC(average_precision_score) 외에,
#   운영 관점에서 상위 위험군 선별 성능을 보기 위한 지표를 직접 정의한다.
# - precision_at_k:
#   예측 확률 상위 k%(여기서는 10%)를 추출했을 때 실제 churn 비율
# - lift_at_k:
#   precision_at_k를 전체 모수의 churn base rate로 나눈 값
#   => 상위 위험군 선별력이 평균 대비 몇 배인지 확인 가능
# ============================================================
def precision_at_k(y_true, y_score, top_ratio=0.10):
    # y_true: 실제 정답(0/1)
    # y_score: 모델 예측 점수/확률
    # top_ratio: 상위 몇 %를 컷오프로 볼지 지정 (기본 10%)
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score)

    # 샘플 수 * top_ratio를 기준으로 상위 추출 개수 k를 계산
    # 최소 1개는 뽑도록 max(1, ...) 처리
    k = max(1, int(np.ceil(len(y_true) * top_ratio)))

    # 예측 점수가 높은 순으로 내림차순 정렬 후 상위 k개 인덱스를 선택
    top_idx = np.argsort(-y_score)[:k]

    # 상위 k개 집합 내 실제 churn 비율 반환
    return y_true[top_idx].mean()

def lift_at_k(y_true, y_score, top_ratio=0.10):
    # 전체 데이터의 평균 churn 비율(base rate)
    base_rate = np.mean(y_true)

    # 상위 k% 기준 precision 계산
    p_at_k = precision_at_k(y_true, y_score, top_ratio)

    # lift = 상위군 precision / 전체 평균 churn 비율
    # base_rate가 0이면 0으로 나눌 수 없으므로 NaN 반환
    return p_at_k / base_rate if base_rate > 0 else np.nan

# ============================================================
# 5) LightGBM 학습
# ------------------------------------------------------------
# - AutoGluon TabularPredictor를 사용해 GBM(LightGBM 계열) 모델을 학습한다.
# - eval_metric은 average_precision(PR-AUC 계열) 기준으로 설정
# - tuning_data는 validation snapshot
# - hyperparameters={"GBM": {}} 로 GBM 계열 모델만 사용
# - presets="best_quality"로 성능 중심 세팅 적용
# - use_bag_holdout=True는 holdout/tuning 데이터를 함께 고려하는 설정
# ============================================================
predictor = TabularPredictor(
    label="target",                 # 예측 대상 컬럼명
    eval_metric="average_precision" # 모델 선택/튜닝 기준 지표
).fit(
    train_data=train_model,         # 학습 데이터
    tuning_data=valid_model,        # 검증 데이터(시간상 test 직전 snapshot)
    hyperparameters={"GBM": {}},    # GBM 계열만 학습
    time_limit=1800,                # 최대 학습 시간(초)
    presets="best_quality",         # 성능 우선 프리셋
    use_bag_holdout=True            # holdout 기반 활용 옵션
)

# ============================================================
# 6) 성능 평가
# ------------------------------------------------------------
# - test snapshot에 대해 최종 성능을 계산한다.
# - 사용 지표:
#   * PR-AUC
#   * Precision@10%
#   * Lift@10%
# - metrics_df는 후속 저장/보고용 요약 테이블이다.
# ============================================================
# 테스트셋 실제 타깃
y_test = test_model["target"]

# 테스트셋 예측 확률
# predict_proba 결과에서 양성 클래스(1번 클래스) 확률을 사용
probs = predictor.predict_proba(test_model).iloc[:, 1]

# PR-AUC 계산
pr_auc = average_precision_score(y_test, probs)

# 상위 10% Precision 계산
prec_10 = precision_at_k(y_test, probs, top_ratio=0.10)

# 상위 10% Lift 계산
lift_10 = lift_at_k(y_test, probs, top_ratio=0.10)

# 성능 요약 데이터프레임 생성
# model: 결과표/저장 시 식별용 모델명
# n_features: 실제 학습에 사용된 최종 feature 개수
metrics_df = pd.DataFrame([{
    "model": "LightGBM_final10",
    "pr_auc": pr_auc,
    "precision_at_10": prec_10,
    "lift_at_10": lift_10,
    "n_features": len(final_features)
}])

# ============================================================
# 7) Type A 기준 Risk_Band 생성
# ------------------------------------------------------------
# - test_data를 기반으로 대시보드용 결과 테이블을 만든다.
# - 예측 확률을 0~100% 단위로 변환한 뒤,
#   확률 순위(rank percentile)를 기준으로 Risk_Band를 구분한다.
# - 현재 기준:
#   * 상위 35%   -> High
#   * 다음 13%   -> Mid   (35% 초과 ~ 48% 이하)
#   * 나머지     -> Low
# - 이 로직은 대시보드용 위험도 구간화 규칙이다.
# ============================================================
dashboard_df = test_data.copy()

# churn 예측 확률을 백분율(%) 형태로 변환
# 소수 둘째 자리까지 반올림하여 대시보드 표시용으로 사용
dashboard_df["churn_probability_pct"] = (probs.values * 100).round(2)

# 확률이 높은 순으로 백분위 순위를 계산
# ascending=False 이므로 높은 확률일수록 더 앞 순위를 가진다.
rank_pct = dashboard_df["churn_probability_pct"].rank(pct=True, ascending=False)

# 백분위 순위를 기준으로 Risk_Band 부여
# 조건 순서:
# - rank_pct <= 0.35 : High
# - rank_pct <= 0.48 : Mid
# - 그 외            : Low
dashboard_df["Risk_Band"] = np.select(
    [rank_pct <= 0.35, rank_pct <= 0.48],
    ["High", "Mid"],
    default="Low"
)

# ============================================================
# 8) 대시보드용 결과 저장
# ------------------------------------------------------------
# - 대시보드에서 사용할 최종 결과 컬럼만 선택한다.
# - 포함 항목:
#   * 식별/시점: user_id, snapshot_date
#   * 예측 결과: churn_probability_pct, Risk_Band
#   * 세그먼트/디바이스 관련 보조 컬럼
#   * 최종 feature들
#   * 실제 정답(target) - 성능 검증/내부 확인용
# - 이후 churn_probability_pct 내림차순으로 정렬하여 저장한다.
# ============================================================
dashboard_columns = [
    "user_id", "snapshot_date", "churn_probability_pct", "Risk_Band",
    "segment_volume", "segment_explore", "segment_taste",
    "freq_tablet", "freq_pc"
] + final_features + ["target"]

# 실제 존재하는 컬럼만 남겨서 에러 방지
dashboard_columns = [col for col in dashboard_columns if col in dashboard_df.columns]

# 최종 대시보드용 데이터셋 생성
# churn_probability_pct가 높은 사용자부터 우선 정렬
dashboard_final = dashboard_df[dashboard_columns].sort_values(
    by="churn_probability_pct", ascending=False
).reset_index(drop=True)

# 대시보드 전체 결과 저장
# DASHBOARD_FULL_PATH: 사용자 단위 위험도/예측 결과 저장 경로
dashboard_final.to_csv(DASHBOARD_FULL_PATH, index=False)

# 성능 지표 요약 저장
# METRICS_PATH: 모델 성능표 저장 경로
metrics_df.to_csv(METRICS_PATH, index=False)